<a href="https://colab.research.google.com/github/gflamiano/SJU-CSC-672-ML/blob/main/HW3/HW3_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW3 Analysis

**What challenges did you encounter when loading and inspecting a raw external dataset from Kaggle compared to pre-packaged datasets, specifically regarding implicit data types like whitespace entries in `TotalCharges`?**

The main challenge was that `TotalCharges` looked complete on the surface but was actually stored as text instead of a number, because 11 rows had a blank space instead of an actual value. These were all customers with `tenure = 0`, meaning they were new and hadn't been billed yet. Since it wasn't a real `NaN`, `df.isna().sum()` didn't catch it, and `.describe()` just quietly left the column out of the numeric summary instead of throwing an error. I only noticed because I compared it to `MonthlyCharges`, which is a similar column and loaded correctly as a float, so the mismatch stood out. With a pre-packaged dataset this kind of thing is already handled for you, so this was a good reminder that with raw data you actually have to check that each column's dtype makes sense instead of just trusting the load.

**How did you determine which columns required missing value imputation versus other data cleaning methods, and what strategy did you choose?**

I started by running `.isna().sum()` on the whole dataset, and it came back showing zero missing values, which turned out to be misleading for the reason above. Once I converted `TotalCharges` to numeric using `errors='coerce'`, the real count showed up: 11 missing values, all in that one column. Since that's only about 0.16% of the data, I went with imputing instead of dropping those rows, since dropping them would throw away 19 other columns of otherwise complete data. I used `SimpleImputer(strategy='median')` instead of mean because `TotalCharges` is right-skewed, so the median is less affected by the small number of high-value outliers in that column.

**Explain how the Scikit-Learn `ColumnTransformer` and `Pipeline` streamline the data preparation process and prevent data leakage across preprocessing and training steps.**

`ColumnTransformer` lets me apply different preprocessing to different columns in one step, scaling the numeric columns and one-hot encoding the categorical ones, instead of splitting everything apart manually and trying to piece it back together correctly. `Pipeline` then wraps that preprocessing together with the classifier so the whole thing works as one object with `.fit()` and `.predict()`. The leakage prevention comes from the fact that calling `pipeline.fit(X_train, y_train)` only lets the scaler and encoder learn from the training data, and then `.predict(X_test)` just applies those already-learned values instead of refitting on the test set. If I had scaled the entire dataset before splitting it, information from the test set would have leaked into training and made the results look better than they actually are.

**How did comparing `StandardScaler` versus `MinMaxScaler` impact your model training, convergence, or final evaluation metrics?**

I used `StandardScaler` for the whole pipeline rather than testing both scalers side by side, so this is more of an expected-outcome answer than something I actually measured. Either scaler should help the model converge faster than leaving the features unscaled, since `SGDClassifier` relies on gradient descent, and gradient descent struggles when features are on very different scales, like `tenure` topping out around 72 versus `TotalCharges` going into the thousands. I chose `StandardScaler` over `MinMaxScaler` because `TotalCharges` has some outliers, and `MinMaxScaler` compresses everything into a strict 0-1 range based on the min and max, so a few extreme values can squeeze the rest of the data into a narrow band. `StandardScaler` doesn't have that issue since it's based on the mean and standard deviation instead of a hard range. I'd expect the difference in final accuracy or ROC-AUC between the two to be fairly small, but since I didn't test it directly I can't say for sure.

**How does configuring `SGDClassifier(loss='log_loss')` differ from standard linear classification models, and what advantages do probability outputs and ROC-AUC metrics provide when evaluating customer churn?**

`SGDClassifier` isn't one fixed model, it's a training method (gradient descent) that can act like different linear models depending on the loss function you give it. Setting `loss='log_loss'` makes it behave like logistic regression, and that's specifically what lets it output real probabilities through `predict_proba()` instead of just a hard class label. Other options like `hinge` loss, which acts more like an SVM, don't give you that. Having probabilities matters for churn prediction because it lets you set your own decision threshold instead of always defaulting to 0.5, so a team could lower the threshold to flag more at-risk customers for outreach even if it means a few more false alarms. ROC-AUC is useful here too because this dataset is imbalanced, with a lot more retained customers than churned ones, so accuracy alone can be misleading. ROC-AUC instead measures how well the model ranks churners above non-churners across every threshold, which gives a more honest sense of whether the model actually learned something useful.

**Based on your model coefficients or feature evaluation, which features show the strongest relationship with customer churn, and what operational business insights do these findings reveal?**

Looking at the coefficients, `Contract_Month-to-month` had a strong positive weight, meaning it pushes predictions toward churn, while `Contract_Two year` had a strong negative weight, pushing toward retention. That makes sense since month-to-month customers can cancel at any time with no penalty. `tenure` was also negatively weighted, so the longer someone has been a customer, the less likely the model is to predict they'll churn. `PaymentMethod_Electronic check` leaned toward churn compared to the automatic payment methods, possibly because manual payments give customers more regular chances to reconsider the service, while autopay just runs in the background without prompting that. From a business standpoint, this points to focusing retention efforts on newer, month-to-month customers, maybe through incentives to switch to annual contracts or autopay. That said, I'd be careful about treating the payment method finding as causal rather than just a correlation, since that would need more testing, like an A/B test, before acting on it too strongly.